# Workshop config (run via `%run ./_config`)

Defines all per-user resource names and the one shared resource (the catalog). Imported by every other notebook so attendees never edit hardcoded strings.

**Shared across all attendees:** `CATALOG`.

**Per attendee** (suffixed by sanitized username): `SCHEMA`, `BASE`, `GENIE_SPACE_NAME`, `KA_NAME`, `AGENT_ENDPOINT`, `APP_NAME`, all volume paths.

In [ ]:
import re
from databricks.sdk import WorkspaceClient

_w = WorkspaceClient()
USER_EMAIL = _w.current_user.me().user_name
# Sanitize the username portion before @: lowercase alphanumeric + underscore, max 20 chars.
_user_local = USER_EMAIL.split('@', 1)[0].lower()
USER_SUFFIX = re.sub(r'[^a-z0-9]+', '_', _user_local).strip('_')[:20] or 'user'

# ---- SHARED ----
CATALOG = "main"  # shared catalog name; override here if your workspace uses a different one

# ---- PER USER ----
SCHEMA = f"cti_{USER_SUFFIX}"
BASE = f"{CATALOG}.{SCHEMA}"

# Volumes live inside the per-user schema
VOL_ADVISORIES = f"/Volumes/{CATALOG}/{SCHEMA}/raw_advisories"
VOL_STIGS = f"/Volumes/{CATALOG}/{SCHEMA}/raw_stigs"
VOL_KA_CORPUS = f"/Volumes/{CATALOG}/{SCHEMA}/ka_corpus"

# Genie space, Knowledge Assistant, agent endpoint, and Databricks App are all per-user
GENIE_SPACE_NAME = f"DISA Threat Intel ({USER_SUFFIX})"
KA_NAME = f"disa-cti-knowledge-{USER_SUFFIX}"[:60]
AGENT_MODEL = f"{BASE}.disa_cti_agent"
AGENT_ENDPOINT = f"disa-cti-agent-{USER_SUFFIX}"[:60]
APP_NAME = f"disa-cti-{USER_SUFFIX}".replace('_', '-')[:30]

# Foundation model endpoints (workspace-wide, not per-user)
LLM_ENDPOINT = "databricks-claude-sonnet-4-6"  # used by the compound agent (smart)
LLM_HAIKU = "databricks-claude-haiku-4-5"      # used by ai_query for parsing (fast)
EMBEDDING_ENDPOINT = "databricks-gte-large-en"

# Per-user state table (no longer shared)
CONFIG_TABLE = f"{BASE}._workshop_config"

print("=" * 60)
print(f"USER          = {USER_EMAIL}")
print(f"USER_SUFFIX   = {USER_SUFFIX}")
print("-" * 60)
print(f"CATALOG       = {CATALOG}    (shared)")
print(f"SCHEMA        = {SCHEMA}")
print(f"BASE          = {BASE}")
print(f"GENIE_SPACE   = {GENIE_SPACE_NAME}")
print(f"KA_NAME       = {KA_NAME}")
print(f"AGENT         = {AGENT_ENDPOINT}")
print(f"APP_NAME      = {APP_NAME}")
print("=" * 60)

In [ ]:
def cfg_get(key, fallback=""):
    """Read a value from the per-user _workshop_config table."""
    try:
        rows = spark.sql(f"SELECT value FROM {CONFIG_TABLE} WHERE key = '{key}'").collect()
        return rows[0].value if rows else fallback
    except Exception:
        return fallback

def cfg_set(key, value):
    """Idempotent upsert into the per-user _workshop_config table."""
    spark.sql(f"""
      CREATE TABLE IF NOT EXISTS {CONFIG_TABLE} (key STRING, value STRING) USING DELTA
    """)
    spark.sql(f"""
      MERGE INTO {CONFIG_TABLE} t
      USING (SELECT '{key}' AS key, '{value}' AS value) s
      ON t.key = s.key
      WHEN MATCHED THEN UPDATE SET value = s.value
      WHEN NOT MATCHED THEN INSERT *
    """)